In [9]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
from llm_zoomcamp.ingest import ingest_faq_data,build_index
from llm_zoomcamp.rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

load_dotenv()

system_instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search return nothing,it's likely an off-topic question.
if you can't answer the question using FAQ don't do it yourself only use the facts from the FAQ database

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [10]:
docs = ingest_faq_data()
index = TextSearchIndex(text_fields=["question", "section", "answer"],
    keyword_fields=["course"],db_path="../data/faq.db")

gemini_client = genai.Client()

config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instructions
)

question = "I just discovered the course. Can i join it?"


In [ ]:
assistant = RAGBase(index = index,llm_client = gemini_client,config = config ,model = "gemini-3.6-flash")
answer = assistant.rag(question)
print(answer)

In [11]:
def search(question,num_results = 5):
    """
    Search the FAQ database for entries matching the given question.

    Args:
         question:search query text to look up in the course FAQ
    """

    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
            question,
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=num_results
    )


search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type":"object",
        "properties" : {
            "question": {
                "type":"string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["question"],
        "additionalProperties": False
    }
)

search_tool = types.Tool(function_declarations=[search_declaration])

In [12]:
tool_config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instructions,
    tools = [search_tool]
)

contents = [
    types.Content(role = "user", parts = [types.Part(text=question)])
]

response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)

response.function_calls

In [ ]:
call = response.function_calls[0]
args = call.args
print(args)
name = call.name
print(name)
print(response.candidates[0].content)

result = search(**args)
print(result)


contents.append(response.candidates[0].content)

contents.append(
    types.Content(
    role = "user",
    parts= [types.Part.from_function_response(
        name=call.name,
        response={"result": result},
    )
    ],
)
)

In [ ]:
response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)
print(response.text)

Agentic Loop

In [13]:
def make_call(call):
    args = call.args

    if call.name == "search":
        result = search(**args)
    else:
        result = {"error": f"Unknown function called {call.name}"}

    return types.Part.from_function_response(name= call.name,response = {"result":result})


In [ ]:
contents = [
    types.Content(role = "user", parts = [types.Part(text=question)])
]

response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)

answer = response.candidates[0].content

contents.append(answer)

function_response_parts = []

for part in answer.parts or []:
    if part.function_call:
        print("function_call:", part.function_call.name,part.function_call.args)
        function_response_parts.append(make_call(part.function_call))

    elif part.text:
        print("Assistant:")
        print(part.text)

if function_response_parts:
    contents.append(types.Content(role="user",parts = function_response_parts))

has_function_calls = len(function_response_parts) > 0

In [ ]:
MODEL = "gemini-3.5-flash"
def agent_loop(instructions,question,model = MODEL,max_iterations=10):
    contents = [
        types.Content(role="user", parts = [types.Part(text=question)])
    ]

    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[search_tool]
    )

    last_answer = ""

    for it in range(1,max_iterations+1):
        print(f"iteration #{it}..")

        response = gemini_client.models.generate_content(
            model = model,
            contents = contents,
            config = config
        )

        model_content = response.candidates[0].content

        contents.append(model_content)

        function_response_parts = []

        for part in model_content.parts or []:
            if part.function_call:
                print("function_call:", part.function_call.name,part.function_call.args)
                function_response_parts.append(make_call(part.function_call))

            elif part.text:
                print("Assistant:")
                print(part.text)
                last_answer = part.text

        if not function_response_parts:
            break

        contents.append(types.Content(role="user",parts=function_response_parts))

    return last_answer

In [ ]:
agent_loop(system_instructions,question = "How do i run Olama locally?")

In [ ]:
agent_loop(system_instructions,question)

In [14]:
from pydantic_ai import Agent

def search(query: str) -> list[dict]:
    """Search the LLM Zoomcamp FAQ for relevant questions and answers.

    Args:
        query: key words from the user's question, in English.
    """
    return index.search(
        query,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
        num_results=5,
    )


agent = Agent(
    "google:gemini-3.6-flash",
    instructions=system_instructions,
    tools=[search]
)

In [15]:
result = await agent.run("How do i run Olama locally?")
print(result.output)

To run Ollama locally, follow these steps:

### 1. Install Ollama
Download and install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) for your operating system:
- **macOS**: Download the `.pkg` file and install it.
- **Windows**: Download the `.msi` file and install it.
- **Linux**: Run the following command in your terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

---

### 2. Run a Model Locally
Once installed, open a terminal and run a model (for example, LLaMA 3):
```bash
ollama run llama3
```

This command will:
- Download the LLaMA 3 model (~4GB).
- Start the model locally.
- Open an interactive chat interface in your terminal.

---

### 3. Test the Ollama Local Server
To verify that the Ollama local server is running, execute:
```bash
curl http://localhost:11434
```
You should get a JSON response showing available models.

---

### 4. (Optional) Access Ollama via Python
Install the Python client:
```bash
pip install ollama
`

In [17]:
from pydantic_ai.messages import ToolCallPart, ToolReturnPart

for msg in result.all_messages():
    for part in msg.parts:
        if isinstance(part, ToolCallPart):
            print("Seeked:", part.args)
        elif isinstance(part, ToolReturnPart):
            print("Found:", [d["question"] for d in part.content])

Seeked: {'query': 'Ollama run locally'}
Found: ['Can I run the course locally instead of Codespaces?', 'Can I run the course locally instead of Codespaces?', 'Can I run the course locally instead of Codespaces?', 'Ollama: How to install Ollama?', 'Ollama: How to install Ollama?']
Seeked: {'query': 'Ollama docker'}
Found: ['Ollama: How to install Ollama?', 'Ollama: How to install Ollama?', 'Ollama: How to install Ollama?', 'Connection refused error when prompting Ollama RAG', 'Connection refused error when prompting Ollama RAG']
Seeked: {'query': 'Ollama'}
Found: ['Ollama: How to install Ollama?', 'Ollama: How to install Ollama?', 'Ollama: How to install Ollama?', 'Connection refused error when prompting Ollama RAG', 'Connection refused error when prompting Ollama RAG']
